# 4. Modelado y Evaluacion de Resultados

Fase 4 de CRISP-DM. Se selecciona la tecnica, se genera el plan de prueba, se entrena y compara modelos, y se evalua el mejor candidato con metricas, residuos, importancia de variables y diagnostico de sesgo/varianza.

## 4.1. Tecnica de Modelado Escogida

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

RANDOM_STATE = 42
DATA_DIR = '../data'
MODEL_CSV = f'{DATA_DIR}/processed/gb_videos_model.csv'
FIG_DIR = '../reports/figures'
os.makedirs(FIG_DIR, exist_ok=True)
sns.set_theme(style='whitegrid')

# Tecnica: regresion supervisada para predecir log1p(views)
print('Tecnica: Regresion supervisada (log1p views) comparando Linear Regression, Decision Tree, Random Forest, Gradient Boosting' + (', XGBoost' if HAS_XGB else ''))

## 4.2. Generacion del Plan de Prueba

In [ ]:
df = pd.read_csv(MODEL_CSV)
print('Shape:', df.shape)

# Variable objetivo: log1p(views) para estabilizar la varianza
X = df.drop(columns='views')
y = np.log1p(df['views'])
print('Features:', X.shape[1])

# Estrategia de prueba: train/test 80/20 + validacion cruzada 5-fold
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 4.3. Construccion del Modelo

In [ ]:
# Definicion y entrenamiento de modelos
modelos = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=15, n_jobs=-1, random_state=RANDOM_STATE
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=5, random_state=RANDOM_STATE
    ),
}
if HAS_XGB:
    modelos['XGBoost'] = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
    )

def evaluar(y_true, y_pred, nombre):
    return {
        'model': nombre,
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

resultados = []
# Baseline: media del target
pred_base = np.full_like(y_test, y_train.mean(), dtype=float)
resultados.append(evaluar(y_test, pred_base, 'Baseline (media)'))

modelos_entrenados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append(evaluar(y_test, pred, nombre))
    modelos_entrenados[nombre] = modelo

metrics = pd.DataFrame(resultados).round(4)
display(metrics.sort_values('MAE'))

# Seleccion del mejor modelo (excluyendo baseline)
mejores = metrics[metrics['model'] != 'Baseline (media)'].sort_values('MAE')
best_name = mejores.iloc[0]['model']
best_model = modelos_entrenados[best_name]
print('Mejor modelo:', best_name)

## 4.4. Evaluacion del Modelo

In [ ]:
# Validacion cruzada del mejor modelo
cv = cross_val_score(best_model, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f'MAE CV: {(-cv).mean():.4f} ± {cv.std():.4f}')

# Prediccion vs Real
y_pred = best_model.predict(X_test)
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.3, edgecolors='none')
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lim, lim, 'r--', lw=2, label='Ideal')
plt.xlabel('Real (log1p views)'); plt.ylabel('Prediccion (log1p views)')
plt.title(f'{best_name}: Prediccion vs Real')
plt.legend(); plt.tight_layout(); plt.savefig(f'{FIG_DIR}/model_predictions.png', dpi=300, bbox_inches='tight'); plt.show()

### 4.4.1 Analisis de residuos

In [ ]:
residuos = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(residuos, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribucion de residuos')
axes[1].scatter(y_pred, residuos, alpha=0.3, edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Prediccion'); axes[1].set_ylabel('Residuo'); axes[1].set_title('Residuos vs Prediccion')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/model_residuals.png', dpi=300, bbox_inches='tight'); plt.show()

### 4.4.2 Importancia de variables

In [ ]:
importancias = pd.Series(best_model.feature_importances_, index=X.columns)
top = importancias.sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, palette='viridis', hue=top.index, legend=False)
plt.title('Top 15 variables mas importantes'); plt.xlabel('Importancia'); plt.ylabel('Variable')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/model_feature_importance.png', dpi=300, bbox_inches='tight'); plt.show()
display(top.to_frame('importancia'))

### 4.4.3 Curva de aprendizaje y diagnostico de sesgo/varianza

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5), random_state=RANDOM_STATE,
)
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, -train_scores.mean(axis=1), 'o-', label='Train')
plt.plot(train_sizes, -test_scores.mean(axis=1), 'o-', label='Validacion')
plt.xlabel('Tamano de entrenamiento'); plt.ylabel('MAE'); plt.title('Curva de aprendizaje'); plt.legend()
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/model_learning_curve.png', dpi=300, bbox_inches='tight'); plt.show()

train_score = best_model.score(X_train, y_train)
test_score = best_model.score(X_test, y_test)
print(f'R2 train: {train_score:.4f} | R2 test: {test_score:.4f}')
if train_score - test_score > 0.1:
    print('Diagnostico: posible sobreajuste (gap train-test alto).')
elif test_score < 0.6:
    print('Diagnostico: posible subajuste (R2 test bajo).')
else:
    print('Diagnostico: buen balance sesgo-varianza.')

### 4.4.4 Insights estrategicos

In [ ]:
print('''
1. Entertainment y Music dominan el volumen, pero no necesariamente el engagement.
2. El engagement_rate y likes_per_view son mejores predictores que los likes absolutos.
3. Los videos publicados en fines de semana pueden mostrar patrones diferentes de consumo.
4. Unos pocos canales concentran la mayoria de apariciones en tendencias.
5. Las regiones con mas actividad tambien presentan mayor dispersion en vistas.
''')